  
  
This script extracts the image from the .zip file downloaded from Planet; prepares the images creating the .png image from the .tif file allowing for the manual annotation wit LabelMe; segments the labelled image; describes that segmented image for an evaluation of their relevance to train the model; culls them to suit the model training requirements; and finally applies the model training. 
  
  

# 1. Path definition
We define the path of the project, and refer the correct python scripts containing the functions that are automatically called when running dependencies. 

In [1]:
#| code-fold: show
import sys
import os
import yaml
import counting_wh
# import counting_wh.wh_utils as wh_utils
# import counting_wh.train as wh_train

from counting_wh import wh_utils
from counting_wh import train

# import counting_wh.wh_utils.planet_utils as planet_utils

# Add the project root to sys.path (adjust as needed)
# sys.path.append(os.path.abspath("Waterholes_project/WaterholeDetection_UN-Handbook"))


# 2. Padded PNG image preparation
We will now transform the example raw tif image into a usable png for future steps. It creates a padded png image to exactly match the size dividable by the stride and tile size.   
Caution that you need to define the config file called "config_train_Drive_UN.yaml" as instructed to make sure it matches your paths and runs everything smoothly.  
Use the prepare() function if you are using the Planet .tif file, and use the prepare_S2() function if you are using the Sentinel 2 file. 

In [2]:
#| code-fold: show 
#For the Sentinel 2 file: 
train.prepare_S2("config_train_Drive_UN.yaml")

Processed 1/1 images


# 3. Manual Labelling
Once the .png image is created, as we are training the model, we need to label the training image. In order to do so, you need to use LabelMe.  
LabelMe is called in your terminal, manually typing "labelme". 

Ensure that the conda environment is activated, which will show on the left of your terminal line. If it says (base), you need to activate the conda environment first by typing "conda activate counting_waterholes". After that, it should show (counting_waterholes) instead of (base), and you should be able to type "labelme" to open the LabelMe GUI.

If labelme command does not work, you might need to install it first using pip. You can do this by running the following command in your terminal:

```
pip install labelme
```

![LabelMe Software Opening](Website_buildup/figures/labelme_wholescreen.png)

Once you have LabelMe open, you can load the padded png image created in the previous step by clicking on "Open" and selecting the image file. The image should be in 


In [3]:
# Load the config file
with open('config_train_Drive_UN.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Extract variables
print(f"The padded image should be in: {config['output_dir']}")

The padded image should be in: training/output


You can then annotate the waterholes on your padded png image. This will create a .json file with the bounding boxes definitions. You will need those labels in the next step to segment the image and the corresponding labels for training purposes.  

In the 'Edit' menu, select 'Create Rectangle' (or press the CTRL + R keys on Windows, or CMD + R on Mac) to start drawing rectangle bounding boxes around each waterhole in the image. Click to set a corner, and click again to create a rectangle around each waterhole. After drawing each rectangle, a dialog box will appear where you can enter the label for the object. Either type a new class label, or select an existing label and click 'OK'. Repeat this process for all waterholes in the image.

You can also edit existing annotations by selection 'Edit Polygons' from the toolbar (or Edit menu). We found this useful for transferring labels between months of imagery where most of the labels were the same, but some waterholes had changed class. To do that, the label file needs to be changed to name of the image you want to annotate, and then it should open when you open that image in LabelMe.

When you have finished annotating all waterholes, save your annotations by selecting 'Save' from the 'File' menu (or pressing CTRL + S on Windows, or CMD + S on Mac). This will save the annotations in a .json file in the same directory as the image, with the same name as the image file.

![LabelMe Annotation example](Website_buildup/figures/labelme_labelling.png)

# 4. Segmentation

Once the whole manual annotation is done, save the outputs, and come back to this script to run the segmentation of the padded png image you just labelled.

In [4]:
#| code-fold: show
#segment the png images
train.segment("config_train_Drive_UN.yaml", train_val_split=0.8)

['training/output/2024-06_mimal_test_S2.json']
Cropping Image: training/output/2024-06_mimal_test_S2.png
[4160 5408    3]
We will have:  1813  images maximum
90.0% of images without labels will be removed


Saving Segments: 100%|██████████| 1813/1813 [00:06<00:00, 269.49it/s]


Skipped 853 images
Empty 603 images


# 5. Evaluation of the segmentation
After the segmentation, we evaluate the result of the segmentation and production of material to train the model using the "train.describe" function. 
Run the bellow cell to describe the results of segmented images. 

In [5]:
#| code-fold: show
#describe the created segmented images: 
train.describe("config_train_Drive_UN.yaml")

Config path: ./training/output/images/train
--------------------------------------------------
| Training dataset statistics                    |
--------------------------------------------------
| Number of original images:                   1 |
| Number of tiles:                          1038 |
| Number of labels:                         1038 |
| Number of individual labels                    |
|   - Total:                               69916 |
|   - Per class:                                   |
|       - Class 0:  19656 (28.11%)              |
|       - Class 3:   9296 (13.30%)              |
|       - Class 2:  24136 (34.52%)              |
|       - Class 1:  16828 (24.07%)              |
| Background Images:   4144 (399.23%)           |
--------------------------------------------------


(1, 1038, 69916, {'0': 19656, '3': 9296, '2': 24136, '1': 16828}, 4144)

# 6. Culling
Before proceeding to the training of the model, we need to apply the cull command which will remove images with no labels until 10% of the training set has no labels. This is a recommended procedure from Yolo to maintain efficient model training parameters. This has to be done post segmentation as we don't know prior the the amount.  

In [6]:
#| code-fold: show
# cull images with no labels until 10% of the training set has no labels.
train.cull_AF("config_train_Drive_UN.yaml")

Analyzing label files in: training/output/labels/train
Looking for corresponding images in: training/output/images/train
Total label files found: 1038
Empty label files found: 148
Non-empty label files: 890
Moving 50 empty label files to maintain 10% ratio

--- SUMMARY ---
Total label files moved: 50
Total image files moved: 50
Remaining total label files: 988
Remaining empty label files: 98
Empty labels now make up 9.92% of the dataset
Empty labels moved to: training/output/labels/moved_empty_labels
Corresponding images moved to: training/output/images/moved_empty_images

SUCCESS: Empty labels now make up 10% or less of the dataset.


# 7. Folder set-up
Now that we have less than 10% of the images unlabelled, we can train the model, but first let's reorganise the folders to be properly used in the model. 

In [7]:
#| code-fold: show
train.reorganize_folders("config_train_Drive_UN.yaml")

Removing existing destination: training/train/val/images
Copying from training/output/images/val to training/train/val/images
Successfully copied to training/train/val/images
Removing existing destination: training/train/train/images
Copying from training/output/images/train to training/train/train/images
Successfully copied to training/train/train/images
Removing existing destination: training/train/val/labels
Copying from training/output/labels/val to training/train/val/labels
Successfully copied to training/train/val/labels
Removing existing destination: training/train/train/labels
Copying from training/output/labels/train to training/train/train/labels
Successfully copied to training/train/train/labels

----- REORGANIZATION SUMMARY -----
Successful operations: 4
Failed operations: 0

All folders were successfully reorganized!

New structure:
- training/train/val/images (copied from training/output/images/val)
- training/train/val/labels (copied from training/output/labels/val)
- tr

#### 9. Model training
The following code provides you with the line of code to run in yolo in order to train a neural network model. Before doing so, modifiy the directory of the yolo model training path in the config file called "config_train_Drive_UN.yaml". 

Then, run this code which provides you with the command to excecute in the cmd terminal of the Yolov5.  
XXXXXXXXXXXXXXX insert screenshot of the yoloy terminal and what it looks like when it's training? Maybe a screen shot and a gif when it trains?   

In [10]:
#| code-fold: show
# train the model
train.train("config_train_Drive_UN.yaml")

Using mps device
Set default tensor type to float32
python ../yolov5/train.py --device mps     --img 416 --batch 8     --workers 0     --epochs 1 --data config_train_Drive_UN.yaml     --weights ../yolov5/yolov5s.pt --save-period 50


train: weights=../yolov5/yolov5s.pt, cfg=, data=config_train_Drive_UN.yaml, hyp=../yolov5/data/hyps/hyp.scratch-low.yaml, epochs=1, batch_size=8, imgsz=416, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=../yolov5/data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=mps, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=0, project=../yolov5/runs/train, name=exp, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=50, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1, artifact_alias=latest, ndjson_console=False, ndjson_file=False
github: up to date with https://github.com/ultralytics/yolov5 ✅
fatal: cannot change to '/Users/scottforrest/Library/CloudStorage/OneDrive-QueenslandUniversityofTechnology/FirstByte': No such file or directory
YOLOv5 🚀 2025-10-21 Python-3.10.18 torch-2.8.0 MPS

hyper

python r"C:\Users\adria\OneDrive - Queensland University of Technology\FirstByte Waterholes WD\yolov5/train.py" --device cpu --img 416 --batch 8 --workers 6 --epochs 100 --data r"C:\Users\adria\OneDrive - Queensland University of Technology\FirstByte Waterholes WD\WaterholeDetection_UN-Handbook\config_train_Drive_UN.yaml" --weights r"C:\Users\adria\OneDrive - Queensland University of Technology\FirstByte Waterholes WD\yolov5\yolov5s.pt" --save-period 50

Comment from AF (08.09): Need to run it and see what it actually prints. Because I ran with the line bellow to train. So need to make sure what we advise to the user as well in term of batch size, img, workers epochs etc... 

In [9]:
python train.py --workers 2 --img 416 --batch 8 --epochs 150 --data config_train_Drive_UN.yaml --weights yolov5s.pt --cache disk

SyntaxError: invalid syntax (3716484853.py, line 1)

In [ ]:
python train.py --workers 2 --img 416 --batch 8 --epochs 150 --data "C:\Users\adria\OneDrive - AdrianoFossati\Documents\MASTER Australia\RA\Waterholes_project\WaterholeDetection_UN-Handbook\config_train_Drive_UN.yaml" --weights yolov5s.pt --cache disk

As we were limited in storage, I ran the model on the SSD. To do so, I changed the cache to the SSD drive we are using as we are limited in the storage available locally. 
Needed to set the project to the SSD which saves the outputs and doesn't increase the C: storage usage. 

Need to actually create the D:/temp and D:/yolo_run on your Drive or external directory as follow:

In [ ]:
set TMPDIR=D:/temp
set TEMP=D:/temp
set TMP=D:/temp
set KMP_DUPLICATE_LIB_OK=TRUE 
python C:/Users/fossatia/Documents/Waterholes_project/yolov5/train.py --device cuda:0 --img 416 --batch 4 --workers 2 --epochs 50 --data C:\Users\fossatia\Documents\Waterholes_project\counting_waterholes\config_train_Drive.yaml --weights C:/Users/fossatia/Documents/Waterholes_project/yolov5/runs/train/exp3/weights/best.pt --cache False --project D:/yolo_runs

The set KMP_DUPLICATE_LIB_OK=TRUE is not recommended on the error command... I tried to google it and it seems we should force an install of the Nomkl using 'conda install nomkl --channel conda-forge'. 
However, by doing so, dependencies might be altered. To be checked. 

End of this script. 